# Bloco 2 — Diagnóstico operacional

Diagnóstico das duas fontes auditadas no Bloco 1. Cada seção responde uma
pergunta operacional com evidência computada. Todo teste de relação reporta
tamanho de efeito. Em amostra grande o p-valor sinaliza significância mesmo para
diferença irrelevante, então o tamanho de efeito é o que sustenta ou nega a
relação.

| Seção | Pergunta | Fonte |
|-------|----------|-------|
| 2.1 | Onde a operação concentra esforço? | D2 |
| 2.2 | Qual o custo da categoria `Miscellaneous`? | D2 |
| 2.3 | Os atributos do D1 têm relação real com os desfechos? | D1 |
| 2.4 | Síntese dos três achados | D1 e D2 |

> **Contexto do Bloco 1.** D2 (`all_tickets_processed_improved_v3.csv`, 47.837
> tickets de TI interno) é dado real classificado em 8 categorias de
> `Topic_group`. D1 (`customer_support_tickets.csv`, 8.469 tickets) tem texto
> livre sintético e metadados categóricos com distribuição uniforme.

## 2.0 — Setup e carga

Parâmetros, imports e recarga autossuficiente dos dois CSVs. O notebook roda do
zero por Restart & Run All. Nenhum estado é importado do notebook do Bloco 1.

In [1]:
# Parâmetros do Bloco 2. Ajuste aqui.
TOP_TERMS = 15          # termos exibidos na leitura de vocabulário (2.2)
MIN_TRIAGE_MISC = 3     # minutos de triagem manual por ticket sem categoria (2.2)
CLOSED = "Closed"       # status do D1 com os desfechos preenchidos (2.3)
REF_CAT = "Purchase"    # categoria focada de referência para contraste (2.2)

from pathlib import Path
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    """Carrega um CSV de solution/datasets/ e retorna o DataFrame.

    Mesma função de carga do Bloco 1. O caminho é resolvido de forma relativa a
    partir do diretório de execução e, como fallback, sobe na árvore até achar
    solution/datasets/. Sem caminho absoluto de máquina.
    """
    candidates = [Path("datasets")]
    candidates += [parent / "solution" / "datasets" for parent in [Path.cwd(), *Path.cwd().parents]]
    data_dir = next((d for d in candidates if d.is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError("Pasta solution/datasets/ não encontrada a partir de " + str(Path.cwd()))
    return pd.read_csv(data_dir / filename, encoding="utf-8", low_memory=False)


# Stopwords e tokenizador do Bloco 1, replicados para autossuficiência.
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "with", "is",
    "are", "was", "be", "please", "hi", "hello", "dear", "thanks", "thank",
    "regards", "best", "kind", "you", "your", "we", "it", "this", "that", "at",
    "as", "by", "from", "re", "pm", "am", "not", "no", "if", "my", "me", "our",
    "us",
}


def tokenize(text: str) -> list[str]:
    """Tokens alfabéticos minúsculos, comprimento >= 3, sem stopwords."""
    return [t for t in re.findall(r"[a-z]+", str(text).lower())
            if len(t) >= 3 and t not in STOPWORDS]


def br(x, dec=2):
    """Formata número no padrão pt-BR: ponto de milhar e vírgula decimal."""
    return f"{x:,.{dec}f}".replace(",", "\u00a7").replace(".", ",").replace("\u00a7", ".")

In [3]:
df1 = load_dataset("customer_support_tickets.csv")
df2 = load_dataset("all_tickets_processed_improved_v3.csv")

EXPECTED = {"D1": 8_469, "D2": 47_837}
for name, df in [("D1", df1), ("D2", df2)]:
    rows = len(df)
    status = "OK" if rows == EXPECTED[name] else "*** AVISO: ESPERADO " + f"{EXPECTED[name]:,}" + " ***"
    print(f"{name}: shape = {df.shape}  [{status}]")

D1: shape = (8469, 17)  [OK]
D2: shape = (47837, 2)  [OK]


## 2.1 — Demanda real (D2)

Pergunta: onde a operação concentra esforço?

In [4]:
vc = df2["Topic_group"].value_counts()
dist2 = pd.DataFrame({
    "contagem": vc,
    "pct": (vc / len(df2) * 100).round(2),
})
dist2["pct_acum"] = dist2["pct"].cumsum().round(2)

print(f"Demanda por Topic_group (D2, {len(df2):,} tickets)")
print(dist2.to_string())

top3 = dist2.head(3)
top3_pct = float(top3["pct"].sum())
top3_names = list(top3.index)
print("\ntop-3:", ", ".join(f"{n} ({p}%)" for n, p in zip(top3_names, top3["pct"])))
print(f"top-3 acumulado: {top3_pct:.2f}%")

Demanda por Topic_group (D2, 47,837 tickets)
                       contagem    pct  pct_acum
Topic_group                                     
Hardware                  13617  28.47     28.47
HR Support                10915  22.82     51.29
Access                     7125  14.89     66.18
Miscellaneous              7060  14.76     80.94
Storage                    2777   5.81     86.75
Purchase                   2464   5.15     91.90
Internal Project           2119   4.43     96.33
Administrative rights      1760   3.68    100.01

top-3: Hardware (28.47%), HR Support (22.82%), Access (14.89%)
top-3 acumulado: 66.18%


**Leitura 2.1.** O esforço concentra em três áreas técnicas. `Hardware`
(28,47%), `HR Support` (22,82%) e `Access` (14,89%) somam 66,18% dos 47.837
tickets. `Hardware` sozinho responde por quase três de cada dez chamados. A
priorização de automação começa por esse top-3. Cada ponto de deflexão nessas
áreas remove o maior volume absoluto de trabalho manual, e a cauda de seis
categorias abaixo de 6% oferece retorno marginal.

## 2.2 — Desperdício de triagem (D2)

Pergunta: qual o custo operacional da categoria `Miscellaneous`?

In [5]:
misc_mask = df2["Topic_group"] == "Miscellaneous"
misc = df2.loc[misc_mask, "Document"]
misc_n = int(misc_mask.sum())
misc_pct = round(misc_n / len(df2) * 100, 2)


def doc_freq(docs) -> Counter:
    """Frequência de documento: em quantos textos cada termo aparece."""
    c = Counter()
    for t in docs:
        c.update(set(tokenize(t)))
    return c


misc_freq = doc_freq(misc)
misc_terms_distintos = len(misc_freq)
misc_top = misc_freq.most_common(TOP_TERMS)
misc_top_term, misc_top_docs = misc_top[0]
misc_top_pct = round(misc_top_docs / misc_n * 100, 1)

# Categoria focada de referência: contraste do termo dominante.
ref = df2.loc[df2["Topic_group"] == REF_CAT, "Document"]
ref_freq = doc_freq(ref)
ref_top_term, ref_top_docs = ref_freq.most_common(1)[0]
ref_top_pct = round(ref_top_docs / len(ref) * 100, 1)

triagem_horas = misc_n * MIN_TRIAGE_MISC / 60
triagem_dias = triagem_horas / 8

print(f"Miscellaneous: {misc_n:,} tickets ({misc_pct}% de {len(df2):,})")
print(f"termos distintos nos {misc_n:,} textos: {misc_terms_distintos:,}")

print(f"\nTop {TOP_TERMS} termos por frequência de documento")
for term, c in misc_top:
    print(f"{term:>14}: {c:>4} docs ({c / misc_n * 100:.1f}%)")

print(f"\nTermo de topo em Miscellaneous: '{misc_top_term}' em {misc_top_pct}% dos docs")
print(f"Termo de topo na categoria focada {REF_CAT}: '{ref_top_term}' em {ref_top_pct}% dos docs")

print(f"\nCusto de triagem a {MIN_TRIAGE_MISC} min por ticket: "
      f"{triagem_horas:.0f} horas ({triagem_dias:.0f} dias úteis de 8h)")

Miscellaneous: 7,060 tickets (14.76% de 47,837)


termos distintos nos 7,060 textos: 5,925

Top 15 termos por frequência de documento
        change: 1739 docs (24.6%)
           add: 1671 docs (23.7%)
       tuesday: 1227 docs (17.4%)
          sent: 1223 docs (17.3%)
          help: 1193 docs (16.9%)
     wednesday: 1187 docs (16.8%)
        friday: 1047 docs (14.8%)
      thursday: 1026 docs (14.5%)
           let:  866 docs (12.3%)
          also:  843 docs (11.9%)
       manager:  836 docs (11.8%)
      engineer:  833 docs (11.8%)
          name:  813 docs (11.5%)
      approval:  803 docs (11.4%)
       details:  782 docs (11.1%)

Termo de topo em Miscellaneous: 'change' em 24.6% dos docs
Termo de topo na categoria focada Purchase: 'administrator' em 80.4% dos docs

Custo de triagem a 3 min por ticket: 353 horas (44 dias úteis de 8h)


**Leitura 2.2.** `Miscellaneous` reúne 7.060 tickets, 14,76% da operação.
O grupo não tem tema dominante. Seu termo de conteúdo mais frequente, `change`,
aparece em 24,6% dos documentos, contra 80,4% de `administrator` na categoria
focada `Purchase`. Os 7.060 textos usam 5.925 termos distintos, e o topo do
vocabulário é genérico (`change`, `add`, `tuesday`, `sent`, `help`). Cada ticket
nesse depósito exige leitura e re-roteamento manual antes de chegar à fila
certa. A 3 minutos de triagem por ticket, os 7.060 chamados consomem cerca de
353 horas, o equivalente a 44 dias úteis de trabalho humano só para descobrir
para onde encaminhar.

## 2.3 — Lacuna de instrumentação (D1)

Pergunta: os atributos do D1 têm relação real com os desfechos?

`First Response Time` e `Time to Resolution` são carimbos de tempo absolutos, não
durações. Ambos ficam dentro de uma janela de cerca de 24 horas em torno de
2023-06-01. Eles entram no teste convertidos para horas desde o instante mais
antigo do conjunto `Closed`. O eta-quadrado é invariante a deslocamento e
escala, então a origem escolhida não altera o tamanho de efeito.

In [6]:
closed = df1[df1["Ticket Status"] == CLOSED].copy()

for col, new in [("First Response Time", "FRT_h"), ("Time to Resolution", "TTR_h")]:
    ts = pd.to_datetime(closed[col])
    closed[new] = (ts - ts.min()).dt.total_seconds() / 3600

CATS = ["Ticket Channel", "Ticket Priority", "Ticket Type"]
OUTCOMES = [
    ("Customer Satisfaction Rating", "Customer Satisfaction Rating"),
    ("First Response Time", "FRT_h"),
    ("Time to Resolution", "TTR_h"),
]


def eta_squared(y, g) -> float:
    """Fração da variância de y explicada pelos grupos g (eta-quadrado)."""
    y = np.asarray(y, dtype=float)
    g = np.asarray(g)
    keep = ~np.isnan(y)
    y, g = y[keep], g[keep]
    grand = y.mean()
    ss_total = ((y - grand) ** 2).sum()
    ss_between = sum(len(y[g == k]) * (y[g == k].mean() - grand) ** 2 for k in pd.unique(g))
    return ss_between / ss_total if ss_total > 0 else 0.0


def magnitude(e: float) -> str:
    """Faixa de Cohen para eta-quadrado."""
    if e < 0.01:
        return "desprezível"
    if e < 0.06:
        return "pequeno"
    if e < 0.14:
        return "médio"
    return "grande"


print(f"Customer Satisfaction Rating: média por grupo ({len(closed):,} tickets Closed)")
for c in CATS:
    m = closed.groupby(c)["Customer Satisfaction Rating"].mean().round(3)
    print(f"\n{c}")
    print(m.to_string())

rows = []
for out_label, out_col in OUTCOMES:
    for c in CATS:
        e = eta_squared(closed[out_col], closed[c])
        rows.append({"desfecho": out_label, "atributo": c,
                     "eta2": round(e, 5), "magnitude": magnitude(e)})
eta_tab = pd.DataFrame(rows)
eta_pivot = eta_tab.pivot(index="desfecho", columns="atributo", values="eta2")
max_eta2 = float(eta_tab["eta2"].max())

print("\nTamanho de efeito (eta-quadrado) por desfecho x atributo")
print(eta_pivot.to_string())
print(f"\nmaior eta-quadrado entre as {len(eta_tab)} combinações: "
      f"{max_eta2:.5f} ({magnitude(max_eta2)})")

# Carimbos de tempo: resolução antes da primeira resposta.
# Duração a partir dos carimbos crus. A subtração usa a mesma origem para os
# dois campos, ao contrário das colunas em horas, que têm origem por campo.
dur_h = (pd.to_datetime(closed["Time to Resolution"])
         - pd.to_datetime(closed["First Response Time"])).dt.total_seconds() / 3600
n_neg = int((dur_h < 0).sum())
pct_neg = round(n_neg / len(closed) * 100, 1)
print(f"\nTime to Resolution menor que First Response Time: "
      f"{n_neg:,} de {len(closed):,} tickets Closed ({pct_neg}%)")

destaque = eta_tab[eta_tab["magnitude"] != "desprezível"]
print("\nPares com efeito não desprezível:",
      "nenhum" if destaque.empty else "\n" + destaque.to_string(index=False))

Customer Satisfaction Rating: média por grupo (2,769 tickets Closed)

Ticket Channel
Ticket Channel
Chat            3.083
Email           2.964
Phone           2.952
Social media    2.969

Ticket Priority
Ticket Priority
Critical    2.959
High        2.983
Low         3.053
Medium      2.977

Ticket Type
Ticket Type
Billing inquiry         3.028
Cancellation request    3.029
Product inquiry         3.017
Refund request          2.935
Technical issue         2.959

Tamanho de efeito (eta-quadrado) por desfecho x atributo
atributo                      Ticket Channel  Ticket Priority  Ticket Type
desfecho                                                                  
Customer Satisfaction Rating         0.00139          0.00062      0.00079
First Response Time                  0.00025          0.00076      0.00053
Time to Resolution                   0.00126          0.00210      0.00031

maior eta-quadrado entre as 9 combinações: 0.00210 (desprezível)

Time to Resolution menor que Fir

In [7]:
# Tabela consolidada de tamanho de efeito, renderizada em markdown.
cols = list(eta_pivot.columns)
lines = ["| desfecho | " + " | ".join(cols) + " |",
         "|" + "---|" * (len(cols) + 1)]
for idx, row in eta_pivot.iterrows():
    lines.append("| " + idx + " | " + " | ".join(f"{row[c]:.5f}" for c in cols) + " |")
display(Markdown("\n".join(lines)))

| desfecho | Ticket Channel | Ticket Priority | Ticket Type |
|---|---|---|---|
| Customer Satisfaction Rating | 0.00139 | 0.00062 | 0.00079 |
| First Response Time | 0.00025 | 0.00076 | 0.00053 |
| Time to Resolution | 0.00126 | 0.00210 | 0.00031 |

**Leitura 2.3.** Os atributos do D1 não explicam os desfechos. Nas nove
combinações de desfecho por atributo o eta-quadrado fica abaixo de 0,0021,
dentro da faixa desprezível de Cohen (< 0,01). A satisfação média por grupo
varia entre 2,95 e 3,08 numa escala de 1 a 5, sem ordenação por prioridade ou
canal. Os campos de tempo agravam o quadro: a diferença `Time to Resolution`
menos `First Response Time` é negativa em 1.365 dos 2.769 tickets `Closed`
(49,3%), o que colocaria a resolução antes da primeira resposta e confirma que
os carimbos de tempo não medem duração real. Nenhum par apresenta efeito não
desprezível, então nada contraria a leitura de independência. O D1 não
instrumenta a operação. A ausência de medição confiável é o primeiro gargalo do
suporte ao cliente.

## 2.4 — Síntese

Consolida os três achados num dict `diagnostico`, preenchido a partir dos
valores computados em 2.1 a 2.3, e o renderiza como tabela markdown.

In [8]:
diagnostico = {
    "demanda": {
        "metrica": (
            f"top-3 Topic_group = {br(top3_pct)}% "
            f"({top3_names[0]} {br(float(dist2.loc[top3_names[0], 'pct']))}%, "
            f"{top3_names[1]} {br(float(dist2.loc[top3_names[1], 'pct']))}%, "
            f"{top3_names[2]} {br(float(dist2.loc[top3_names[2], 'pct']))}%)"
        ),
        "leitura": "O esforço concentra em três áreas técnicas que somam dois terços do volume.",
    },
    "desperdicio": {
        "metrica": (
            f"Miscellaneous = {br(misc_n, 0)} tickets ({br(misc_pct)}%), "
            f"termo de topo '{misc_top_term}' em {br(misc_top_pct, 1)}% dos docs "
            f"contra {br(ref_top_pct, 1)}% em {REF_CAT}"
        ),
        "leitura": (
            f"Um em cada sete tickets cai num depósito sem tema dominante e custa "
            f"cerca de {br(triagem_horas, 0)} horas de re-roteamento manual."
        ),
    },
    "instrumentacao": {
        "metrica": (
            f"eta-quadrado no máximo {br(max_eta2, 5)} nas {len(eta_tab)} combinações; "
            f"{br(pct_neg, 1)}% de durações negativas"
        ),
        "leitura": "Os atributos do D1 não explicam os desfechos e os carimbos de tempo não medem duração.",
    },
}

print(json.dumps(diagnostico, ensure_ascii=False, indent=2))

linhas = ["| Achado | Métrica | Leitura |", "|--------|---------|---------|"]
for achado, cell in diagnostico.items():
    linhas.append(f"| {achado} | {cell['metrica']} | {cell['leitura']} |")
md_diag = "\n".join(linhas)
print("\n" + md_diag)
display(Markdown(md_diag))

{
  "demanda": {
    "metrica": "top-3 Topic_group = 66,18% (Hardware 28,47%, HR Support 22,82%, Access 14,89%)",
    "leitura": "O esforço concentra em três áreas técnicas que somam dois terços do volume."
  },
  "desperdicio": {
    "metrica": "Miscellaneous = 7.060 tickets (14,76%), termo de topo 'change' em 24,6% dos docs contra 80,4% em Purchase",
    "leitura": "Um em cada sete tickets cai num depósito sem tema dominante e custa cerca de 353 horas de re-roteamento manual."
  },
  "instrumentacao": {
    "metrica": "eta-quadrado no máximo 0,00210 nas 9 combinações; 49,3% de durações negativas",
    "leitura": "Os atributos do D1 não explicam os desfechos e os carimbos de tempo não medem duração."
  }
}

| Achado | Métrica | Leitura |
|--------|---------|---------|
| demanda | top-3 Topic_group = 66,18% (Hardware 28,47%, HR Support 22,82%, Access 14,89%) | O esforço concentra em três áreas técnicas que somam dois terços do volume. |
| desperdicio | Miscellaneous = 7.060 tickets (14

| Achado | Métrica | Leitura |
|--------|---------|---------|
| demanda | top-3 Topic_group = 66,18% (Hardware 28,47%, HR Support 22,82%, Access 14,89%) | O esforço concentra em três áreas técnicas que somam dois terços do volume. |
| desperdicio | Miscellaneous = 7.060 tickets (14,76%), termo de topo 'change' em 24,6% dos docs contra 80,4% em Purchase | Um em cada sete tickets cai num depósito sem tema dominante e custa cerca de 353 horas de re-roteamento manual. |
| instrumentacao | eta-quadrado no máximo 0,00210 nas 9 combinações; 49,3% de durações negativas | Os atributos do D1 não explicam os desfechos e os carimbos de tempo não medem duração. |